In [1]:
import numpy as np
from tqdm import tqdm

The neural network we are going to recreate is made up of only one neuron. Basically, it is just a perceptron and we are going to try to optimize that.

An Important insight is that these neural networks cannot really generate negative values here. It is a leaky abstraction. Because of the fact that we are using relu activation we are unable to generate a negative value. Because it just clips it down to a negative value. And, also the gradient is such that if the activations are negative. Then, the gradient should be zero.

In [2]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([11, 13, 19, 18, 21, -2, 0, -3, 9, -2, -1, 5])

In [1055]:
weight = np.array([1, 1, 1])
bias = np.array(1)

In [1056]:
N = len(X)
learning_rate = 0.001

In [1060]:
for i in tqdm(range(100000)):
    activations = np.maximum(0, X @ weight + bias)
    error = activations - Y

    error[activations <= 0] = 0


    gradient_w = 2/N *  np.sum( error * X.T , axis=1)
    gradient_b = 2/N *  np.sum(error)





    weight = weight - learning_rate*gradient_w
    bias = bias - learning_rate*gradient_b

    # if i % 100 == 0:
    # print(f"Weight = {weight} Bias = {bias}")

100%|██████████| 100000/100000 [00:02<00:00, 36883.38it/s]


In [1061]:
print(f"Weight = {weight} Bias = {bias}")

Weight = [ 4.00000012  1.00000004 -1.00000001] Bias = 4.9999995530323265


Another important insight is that if it jumps too far in the negative, it is hard for it to recover because the gradient then just collapses immmediately.

In [13]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([[123], [129], [243], [168], [201], [169], [357], [74], [322], [250], [202], [707]])


In [22]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

In [23]:
# for i in range(12):
#     f0 = W1 @ X[i] + B1
#     h1 = np.maximum(f0, 0)
#     print(W2 @ h1 + B2)


In [14]:
X = X.reshape(12, 3, 1)
Y = Y.reshape(12, 1, 1)

In [15]:
class NeuralNetwork():

    def __init__(self, X, Y):

        # np.random.randint(1, 5, size=(4, 3))

        W0 = np.random.uniform(0, 0.3, size=(1, 100, 3))

        B0 = np.random.uniform(0, 0.3, size=(1, 100, 1))

        W1 = np.random.uniform(0, 0.3, size=(1, 100, 100))

        B1 = np.random.uniform(0, 0.3, size=(1, 100, 1))

        W2 = np.random.uniform(0, 0.3, size=(1, 1, 100))

        B2 = np.random.uniform(0, 0.3, size=(1, 1, 1))

        # W0 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
        # B0 = np.array([[1],
        #                [1],
        #                [0],
        #                [1]])

        # W1 = np.array([[1, 2, 3, 4]])
        # B1 = np.array([[2]])

        self.W = [W0, W1, W2]

        self.B = [B0, B1, B2]

        self.X = X

        self.Y = Y

        self.f = []

        self.h = []

        self.gradient_f = []

        self.gradient_W = []

        self.gradient_B = []

    def forward_pass(self):

        f0 = self.W[0] @ self.X + self.B[0]

        h1 = np.maximum(f0, 0)

        f1 = self.W[1] @ h1 + self.B[1]

        h2 = np.maximum(f1, 0)

        f2 = self.W[2] @ h2 + self.B[2]

        self.f = [f0, f1, f2]

        self.h = [0, h1, h2]

    def get_gradients(self, x, y):

        diagonal_matrix_0 = np.diag(self.f[0].flatten())
        np.fill_diagonal(diagonal_matrix_0, np.maximum(diagonal_matrix_0.diagonal(), 0))

        diagonal_matrix_1 = np.diag(self.f[1].flatten())
        np.fill_diagonal(diagonal_matrix_1, np.maximum(diagonal_matrix_1.diagonal(), 0))

        gradient_f2 = 2 * (self.f[2] - y)
        gradient_f1 = gradient_f2 @ self.W[2] @ diagonal_matrix_1

        gradient_f0 = gradient_f1 @ self.W[1] @ diagonal_matrix_0

        gradient_B0 = gradient_f0.T
        gradient_B1 = gradient_f1.T
        gradient_B2 = gradient_f2.T

        gradient_W0 = (x @ gradient_f0).T
        gradient_W1 = (self.h[1] @ gradient_f1).T
        gradient_W2 = (self.h[2] @ gradient_f2).T

        self.gradient_f.append( [gradient_f0, gradient_f1, gradient_f2] )
        self.gradient_W.append( [gradient_W0, gradient_W1, gradient_W2] )
        self.gradient_B.append( [gradient_B0, gradient_B1, gradient_B2] )

    def gradient_descent(self, learning_rate= 0.00000001):

        gradient_B0 = np.mean([sublist[0] for sublist in self.gradient_B])
        gradient_B1 = np.mean([sublist[1] for sublist in self.gradient_B])
        gradient_B2 = np.mean([sublist[2] for sublist in self.gradient_B])

        gradient_W0 = np.mean([sublist[0] for sublist in self.gradient_W])
        gradient_W1 = np.mean([sublist[1] for sublist in self.gradient_W])
        gradient_W2 = np.mean([sublist[2] for sublist in self.gradient_W])

        self.B[0] = self.B[0] - learning_rate*gradient_B0
        self.B[1] = self.B[1] - learning_rate*gradient_B1
        self.B[2] = self.B[2] - learning_rate*gradient_B2

        self.W[0] = self.W[0] - learning_rate*gradient_W0
        self.W[1] = self.W[1] - learning_rate*gradient_W1
        self.W[2] = self.W[2] - learning_rate*gradient_W2

    def calculate_mse(self):
        outputs = []
        for i in range(len(self.X)):
            self.forward_pass(self.X[i])
            outputs.append(self.f[2])

        outputs = np.array(outputs)

        return np.mean((self.Y - outputs) ** 2)

    def train(self, epochs = 100):

        for _ in tqdm(range(epochs)):

            self.gradient_f = []

            self.gradient_W = []

            self.gradient_B = []

            for i in range(len(X)):

                self.forward_pass(X[i])

                self.get_gradients(X[i], Y[i])

            self.gradient_descent()

In [16]:
nn = NeuralNetwork(X, Y)


In [17]:
nn.forward_pass()

In [18]:
gradient_f2 = 2 * (nn.f[2] - nn.Y)

In [19]:
gradient_f2

array([[[189.57727809]],

       [[240.80463849]],

       [[370.58475008]],

       [[362.89754709]],

       [[433.76309528]],

       [[288.0041015 ]],

       [[423.2224164 ]],

       [[253.96271323]],

       [[539.87712169]],

       [[461.79447247]],

       [[473.74545728]],

       [[791.14566567]]])

In [20]:
nn.f[1].shape

(12, 100, 1)

In [21]:
diagonal_matrices_1 = []
for i in range(12):
    diagonal_matrix_1 = np.diag(nn.f[1][i].flatten())
    np.fill_diagonal(diagonal_matrix_1, np.maximum(diagonal_matrix_1.diagonal(), 0))

    diagonal_matrices_1.append(diagonal_matrix_1)

diagonal_matrices_1 = np.array(diagonal_matrices_1)

In [22]:
diagonal_matrices_0 = []
for i in range(12):
    diagonal_matrix_0 = np.diag(nn.f[0][i].flatten())
    np.fill_diagonal(diagonal_matrix_0, np.maximum(diagonal_matrix_0.diagonal(), 0))

    diagonal_matrices_0.append(diagonal_matrix_0)

diagonal_matrices_0 = np.array(diagonal_matrices_0)

In [23]:
gradient_f1 = gradient_f2 @ nn.W[2] @ diagonal_matrices_1
gradient_f0 = gradient_f1 @ nn.W[1] @ diagonal_matrices_0

In [27]:
gradient_B2 = np.mean( np.transpose(gradient_f2, (0, 2, 1)), axis=0)[np.newaxis, :, :]
gradient_B1 = np.mean( np.transpose(gradient_f1, (0, 2, 1)), axis=0)[np.newaxis, :, :]
gradient_B0 = np.mean( np.transpose(gradient_f0, (0, 2, 1)), axis=0)[np.newaxis, :, :]

In [26]:
gradient_f0.shape

(12, 1, 100)

In [30]:
(nn.B[0] - gradient_B0).shape

(1, 100, 1)

In [28]:
gradient_B0.shape, nn.B[0].shape

((1, 100, 1), (1, 100, 1))

In [124]:
(nn.B[0] - gradient_B0).shape

(1, 100, 1)

In [117]:
gradient_B0.shape, gradient_B1.shape, gradient_B2.shape

((12, 100, 1), (12, 100, 1), (12, 1, 1))

In [121]:
np.mean(gradient_B0, axis=0).shape

(100, 1)

In [119]:
nn.B[0].shape, nn.B[1].shape, nn.B[2].shape

((1, 100, 1), (1, 100, 1), (1, 1, 1))

In [113]:
gradient_B2.shape

(12, 1, 1)

In [115]:
gradient_f2.shape

(12, 1, 1)

In [109]:
gradient_B2.shape

(1, 1, 12)

In [110]:
gradient_f2.shape

(12, 1, 1)

In [105]:
gradient_f0.shape

(12, 1, 100)

In [107]:
nn.f[0].shape

(12, 100, 1)

In [100]:
gradient_f1.shape

(12, 1, 100)

In [103]:
nn.f[1].shape

(12, 100, 1)

In [ ]:
gradient_f1 = gradient_f2 @ .W[2] @ diagonal_matrix_1

In [54]:
gradient_f2.shape

(12, 1, 1)

In [50]:
nn.f[2].shape

(12, 1, 1)

In [61]:
nn.train(epochs=10000)

100%|██████████| 10000/10000 [05:21<00:00, 31.12it/s]


In [62]:
nn.calculate_mse()

1035.3760329601482

In [94]:
outputs = []
for i in range(len(X)):
    nn.forward_pass(X[i])
    outputs.append(nn.f[2])

outputs = np.array(outputs)

In [63]:
for i in range(12):
    nn.forward_pass(X[i])
    # print(nn.f[0])
    # print(nn.f[1])
    print(nn.f[2], Y[i])

[[107.82868701]] [[123]]
[[129.59853463]] [[129]]
[[238.79481662]] [[243]]
[[194.99483847]] [[168]]
[[238.62129468]] [[201]]
[[173.28548309]] [[169]]
[[326.2866323]] [[357]]
[[107.54213537]] [[74]]
[[347.87806891]] [[322]]
[[282.35895723]] [[250]]
[[260.41558767]] [[202]]
[[653.93108144]] [[707]]


In [ ]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

The truth is there is no proof of convergence for deep neural networks because backpropogation doesn't find the global minimum and is often stuck in a local minima. How would we arrange these matrices in such a way that it would find the global minima is still quite a mystery to me.

It is just super easy to get stuck in a local minima and you might not ever be able to get out of that local minima because you are modifying all of those weights at the same time.